In [1]:
!pip install statsbombpy

import pandas as pd
import numpy as np
import statsbombpy.sb as sb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 923.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.4 MB/s eta 0:00:00


### As the Aggregated Dataset will become too huge otherwise, I will only Focus my Modeling Efforts on Select Teams from among Europe's Top 5 Football Leagues for now

### Even with that all, we will be focusing on Defenders - the Players that form the Defensive Unit of a Team, as Hypothetically for this Scenario, we are recruiting to shore up our Wing-Back and Central Defender positions, one a piece, respectively, this Summer



---



In [2]:
# Combining the Latest Seasons of each of Europe's Top 5 Leagues (in Recording at Statsbomb) into one Master Dataset

def load_multi_competition_events(
    target_leagues: list[str] = None,
    season_names: list[str] = None,
    target_teams: list[str] = None
) -> pd.DataFrame:
    """
    Fetches and concatenates match event streams across multiple target competitions
    for given seasons into a single unified Pandas DataFrame,
    optionally filtered by a list of target teams.
    """

    # Map user-provided team names to StatsBomb's actual team names in the 'team' column.
    # This map goes from the canonical name (what the user passes in `target_teams`)
    # to a list of all possible StatsBomb names it could map to.
    # This allows for flexible matching against the 'team' column in the data.
    target_team_to_sb_names_map = {
        "Manchester United": ["Manchester United"],
        "Liverpool": ["Liverpool"],
        "Arsenal": ["Arsenal"],
        "Chelsea": ["Chelsea"],
        "Manchester City": ["Manchester City"],
        "Real Madrid": ["Real Madrid", "Real Madrid CF"],
        "FC Barcelona": ["Barcelona", "FC Barcelona"], # Including FC Barcelona as user input too
        "Barcelona": ["Barcelona", "FC Barcelona"], # User input 'Barcelona' can match 'FC Barcelona' in data
        "Atletico Madrid": ["Atlético Madrid"], # Use the accented version from StatsBomb
        "Bayern Munich": ["Bayern Munich", "FC Bayern München"], # User input 'Bayern Munich' can match 'FC Bayern München' in data
        "Borussia Dortmund": ["Borussia Dortmund"],
        "Inter Milan": ["Inter"], # StatsBomb typically uses 'Inter'
        "AC Milan": ["Milan"], # StatsBomb typically uses 'Milan'
        "Juventus": ["Juventus"],
        "Napoli": ["Napoli"],
        "Paris Saint-Germain": ["Paris Saint-Germain", "PSG"],
    }

    if target_leagues is None:
        target_leagues = [
            "La Liga",
            "Premier League",
            "1. Bundesliga",
            "Serie A",
            "Ligue 1"
        ]
    if season_names is None:
        season_names = ["2020/2021"] * len(target_leagues) # Default to a single season if not specified

    if len(target_leagues) != len(season_names):
        raise ValueError("The number of target leagues must match the number of season names.")

    # 1. Fetch available open competitions index
    comps = sb.competitions()

    all_league_events = []

    # 2. Iterate through each league and its corresponding season
    for i, league_name in enumerate(target_leagues):
        current_season_name = season_names[i]

        # Filter for the specific league and season
        matching_comps = comps[
            (comps['competition_name'] == league_name) &
            (comps['season_name'] == current_season_name) &
            (comps['competition_gender'] == 'male')
        ]

        if matching_comps.empty:
            print(f"No matching competition found for {league_name} ({current_season_name}). Skipping.")
            continue

        # Assuming there's only one match per league/season/gender combination
        row = matching_comps.iloc[0]
        comp_id = row['competition_id']
        s_id = row['season_id']

        print(f"Fetching events for {league_name} ({current_season_name})...")
        try:
            # Fetch all competition events directly using sb.competition_events
            events_df = sb.competition_events(
                country = row['country_name'],
                division = league_name,
                season = current_season_name,
                gender = "male"
            )

            # CRITICAL: Check if 'team' column exists. If not, skip this data entirely.
            if 'team' not in events_df.columns:
                print(f"  -> Error: 'team' column not found in events for {league_name} ({current_season_name}). Skipping data for this league/season.")
                continue

            # Attach explicit league/season metadata tags
            events_df['league_name'] = league_name
            events_df['season_name'] = current_season_name

            # Filter by target teams if provided
            if target_teams:
                # Build a set of all possible StatsBomb team names corresponding to the target teams
                all_possible_sb_names = set()
                for team_input in target_teams:
                    all_possible_sb_names.update(target_team_to_sb_names_map.get(team_input, [team_input]))

                # Filter events where the 'team' is in the set of possible names
                initial_event_count = len(events_df)
                events_df = events_df[events_df['team'].isin(all_possible_sb_names)]

                if not events_df.empty:
                    print(f"  -> Filtered {initial_event_count - len(events_df):,} events by teams. Keeping data for teams: {', '.join(target_teams)}")
                else:
                    print(f"  -> No events found for target teams {', '.join(target_teams)} in {league_name} ({current_season_name}). Skipping.")
                    continue # Skip this league/season if no target team events were found after filtering

            all_league_events.append(events_df)
            print(f"  -> Successfully loaded {len(events_df):,} events.")

        except Exception as e:
            print(f"  -> Skipping {league_name} (Error loading events): {e}")

    # 3. Concatenate all event Dataframes into a single Master Table
    if all_league_events:
        master_df = pd.concat(all_league_events, ignore_index = True)
        print(f"\nMaster Dataset Successfully Built! Shape: {master_df.shape}")
        return master_df
    else:
        print("No event streams loaded.")
        return pd.DataFrame()

In [3]:
# Executing it: the Multi-League Batch Load to get our Aggregated "Master Dataset"!
master_df = load_multi_competition_events(
    target_leagues = ["La Liga", "Premier League", "Serie A", "1. Bundesliga"],
    season_names = ["2020/2021", "2015/2016", "2015/2016", "2023/2024"],
    target_teams = ["Real Madrid", "Barcelona", "Manchester United", "Liverpool", "Arsenal", "Inter", "Milan", "Bayern Munich", "Borussia Dortmund"]
)

# Getting the Defensive Players' Select Dataset

master_df = master_df[master_df['position'].isin(['Left Center Back',
                                      'Right Center Back',
                                      'Left Back',
                                      'Right Back',
                                      'Left Wing Back',
                                      'Right Wing Back',
                                      'Center Back'])].copy()

# Note: Serie A (Italy) latest: 2015/2016, 1. Bundesliga (Germany) latest: 2023/2024, Ligue 1 (France) latest: 2015/2016

Fetching events for La Liga (2020/2021)...
  -> Filtered 49,225 events by teams. Keeping data for teams: Real Madrid, Barcelona, Manchester United, Liverpool, Arsenal, Inter, Milan, Bayern Munich, Borussia Dortmund
  -> Successfully loaded 89,805 events.
Fetching events for Premier League (2015/2016)...
  -> Filtered 1,084,820 events by teams. Keeping data for teams: Real Madrid, Barcelona, Manchester United, Liverpool, Arsenal, Inter, Milan, Bayern Munich, Borussia Dortmund
  -> Successfully loaded 228,953 events.
Fetching events for Serie A (2015/2016)...
  -> No events found for target teams Real Madrid, Barcelona, Manchester United, Liverpool, Arsenal, Inter, Milan, Bayern Munich, Borussia Dortmund in Serie A (2015/2016). Skipping.
Fetching events for 1. Bundesliga (2023/2024)...
  -> Filtered 129,796 events by teams. Keeping data for teams: Real Madrid, Barcelona, Manchester United, Liverpool, Arsenal, Inter, Milan, Bayern Munich, Borussia Dortmund
  -> Successfully loaded 7,969 e

In [4]:
# Narrowing it down to Primarily Defensive Metrics
# Also Removing any which had Minimal or Unneeded Values for our Analyses & Modeling

master_df = master_df[['type', 'player', 'position', 'team', 'league_name', 'season_name', 'possession_team', 'possession',
                        'timestamp',
                        'under_pressure',
                        'block_save_block', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_other', 'clearance_right_foot',
                        'duel_type', 'duration', 'duel_outcome',
                        'foul_committed_type', 'foul_committed_card', 'foul_committed_penalty',
                        'interception_outcome',
                        'minute',
                        'miscontrol_aerial_won',
                        'out',
                        'location', 'pass_body_part', 'pass_height', 'pass_angle', 'pass_inswinging', 'pass_outswinging', 'pass_length', 'pass_miscommunication', 'pass_no_touch', 'pass_outcome', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_aerial_won', 'pass_recipient', 'pass_straight', 'pass_switch', 'pass_technique', 'pass_type',
                        'play_pattern',
                        'player_off_permanent'
                        ]].copy()

master_df.sample(4)

,type,player,position,team,league_name,season_name,possession_team,possession,timestamp,under_pressure,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,duel_type,duration,duel_outcome,foul_committed_type,foul_committed_card,foul_committed_penalty,interception_outcome,minute,miscontrol_aerial_won,out,location,pass_body_part,pass_height,pass_angle,pass_inswinging,pass_outswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_aerial_won,pass_recipient,pass_straight,pass_switch,pass_technique,pass_type,play_pattern,player_off_permanent
258487,Carry,Faustino Marcos Alberto Rojo,Left Back,Manchester United,Premier League,2015/2016,Manchester United,78,00:40:04.524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[94.7, 4.2]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.160654,NaN,NaN,NaN,NaN,NaN,40,NaN,NaN,"[94.3, 3.1]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,From Throw In,NaN
283431,Pressure,Guillermo Varela Olivera,Right Back,Manchester United,Premier League,2015/2016,Arsenal,73,00:35:55.883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.469518,NaN,NaN,NaN,NaN,NaN,35,NaN,NaN,"[32.0, 49.0]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Play,NaN
251820,Carry,Ignacio Monreal Eraso,Left Back,Arsenal,Premier League,2015/2016,Arsenal,138,00:26:02.172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[64.6, 11.6]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.201790,NaN,NaN,NaN,NaN,NaN,71,NaN,NaN,"[57.0, 10.7]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Play,NaN
214454,Ball Receipt*,Kieran Gibbs,Left Back,Arsenal,Premier League,2015/2016,Arsenal,37,00:16:50.574,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,NaN,NaN,"[43.6, 7.1]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,From Keeper,NaN


In [13]:
# Saving the Master DataFrame for the Main Analytical & Modeling part of the Workflow/Project
master_df.to_csv('master_df.csv')